# 01. Perfilamiento de la fuente primaria: EVA agrícola

## 1. Contexto y objetivo

Este notebook realiza el perfilamiento exploratorio y reproducible de la fuente
primaria de datos del proyecto ETL orientado al análisis de la producción
agrícola en Colombia, en el contexto del Objetivo de Desarrollo Sostenible 2
(ODS 2: Hambre Cero).

La fuente corresponde a las Evaluaciones Agropecuarias Municipales (EVA),
con información agrícola de Colombia para el período 2019–2025.

El propósito del análisis es comprender la estructura, la cobertura y las
características de los datos antes de incorporarlos al pipeline automatizado.

### Objetivos específicos

1. Examinar la estructura general del archivo y sus tipos de datos.
2. Identificar valores nulos, registros duplicados y posibles inconsistencias.
3. Analizar la distribución y el comportamiento de las métricas agrícolas.
4. Verificar la consistencia temporal y geográfica de los registros.
5. Examinar las relaciones entre los atributos que describen los cultivos.
6. Evaluar la granularidad de los registros y proponer una clave candidata.
7. Documentar los hallazgos que deberán considerarse en las transformaciones
   y validaciones de calidad posteriores.

### Alcance

Este notebook corresponde a una etapa de perfilamiento y exploración
de la fuente original. No modifica el archivo fuente ni sustituye las
validaciones operativas que posteriormente se implementarán con
Great Expectations y Apache Airflow.

Los hallazgos exploratorios servirán como evidencia para definir reglas
de calidad, criterios de integración y decisiones del modelo dimensional.

## 2. Metodología

El perfilamiento se realizará mediante análisis descriptivo y verificaciones
de consistencia sobre los datos originales.

Se distinguirán tres tipos de resultados:

- **Observaciones:** características verificables del archivo, como cantidad
  de registros, valores faltantes y distribución de atributos.
- **Posibles anomalías:** valores que requieren investigación, como áreas
  cosechadas superiores a las sembradas o métricas negativas.
- **Decisiones de diseño:** criterios que deberán confirmarse y documentarse
  antes de implementar las reglas definitivas del pipeline.

Una anomalía estadística no se considerará automáticamente un error.
Su clasificación dependerá del significado del atributo, las reglas del
negocio y la documentación de la fuente.

In [5]:
from pathlib import Path
import re
import json
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Rutas del proyecto
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd()

# Si Jupyter se ejecuta desde la carpeta notebooks/,
# subir un nivel para llegar a la raíz del proyecto.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "source"
    / "Evaluaciones_Agropecuarias_Municipales_–_EVA._2019_-_2025._"
    "Base_Agrícola_20260905.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "profiling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Directorio del proyecto:", PROJECT_ROOT)
print("Archivo fuente:", CSV_PATH)
print("Directorio de resultados:", OUTPUT_DIR)

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo fuente:\n{CSV_PATH}\n\n"
        "Verifica el nombre del archivo, su extensión y su ubicación "
        "dentro de data/raw/."
    )

Directorio del proyecto: c:\Users\Monica\Desktop\ETL\ODS2
Archivo fuente: c:\Users\Monica\Desktop\ETL\ODS2\data\source\Evaluaciones_Agropecuarias_Municipales_–_EVA._2019_-_2025._Base_Agrícola_20260905.csv
Directorio de resultados: c:\Users\Monica\Desktop\ETL\ODS2\outputs\profiling


In [6]:
COLUMNAS_METRICAS = [
    "Área sembrada",
    "Área cosechada",
    "Producción",
    "Rendimiento",
]

COLUMNAS_GEOGRAFICAS = [
    "Código Dane departamento",
    "Departamento",
    "Código Dane municipio",
    "Municipio",
]

COLUMNAS_AGRICOLAS = [
    "Grupo cultivo",
    "Subgrupo",
    "Cultivo",
    "Desagregación cultivo",
    "Código del cultivo",
    "Nombre científico del cultivo",
]

COLUMNAS_TEMPORALES = [
    "Año",
    "Periodo",
]

COLUMNAS_CATEGORICAS = [
    "Ciclo del cultivo",
    "Estado físico del cultivo",
]

COLUMNAS_ESPERADAS = (
    COLUMNAS_METRICAS
    + COLUMNAS_GEOGRAFICAS
    + COLUMNAS_AGRICOLAS
    + COLUMNAS_TEMPORALES
    + COLUMNAS_CATEGORICAS
)

print("Cantidad de columnas esperadas:", len(COLUMNAS_ESPERADAS))

Cantidad de columnas esperadas: 18


## 3. Carga y revisión inicial

El archivo se cargará preservando inicialmente sus valores como texto.
Esta decisión evita conversiones automáticas que podrían alterar códigos
geográficos, códigos de cultivos o números que utilicen convenciones
regionales de separadores decimales.

Por ejemplo, los códigos DANE deben conservarse como texto para evitar
la pérdida de ceros iniciales y prevenir que sean tratados como cantidades
numéricas.

Las conversiones se realizarán posteriormente, de forma explícita,
sobre las columnas que representan métricas cuantitativas.

In [7]:
df = pd.read_csv(
    CSV_PATH,
    encoding="utf-8-sig",
    dtype=str,
    low_memory=False,
)

print(f"Cantidad de registros: {len(df):,}")
print(f"Cantidad de columnas: {len(df.columns):,}")

display(df.head())

Cantidad de registros: 166,732
Cantidad de columnas: 18


,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Grupo cultivo,Subgrupo,Cultivo,Desagregación cultivo,Año,Periodo,Área sembrada,Área cosechada,Producción,Rendimiento,Ciclo del cultivo,Estado físico del cultivo,Código del cultivo,Nombre científico del cultivo
0,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021A,"8,00","8,00","128,00","16,00",Transitorio,En fresco,1050600,Apium graveolens
1,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021B,"8,00","8,00","152,00","19,00",Transitorio,En fresco,1050600,Apium graveolens
2,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022A,"8,00","8,00","127,44","15,93",Transitorio,En fresco,1050600,Apium graveolens
3,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022B,"8,00","8,00","152,00","19,00",Transitorio,En fresco,1050600,Apium graveolens
4,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2023,2023A,"8,00","7,00","133,00","19,00",Transitorio,En fresco,1050600,Apium graveolens


In [8]:
print("Información general del DataFrame:")
df.info()

print("\nColumnas encontradas:")
for columna in df.columns:
    print(f"- {columna}")

faltantes_columnas = sorted(
    set(COLUMNAS_ESPERADAS) - set(df.columns)
)

columnas_no_esperadas = sorted(
    set(df.columns) - set(COLUMNAS_ESPERADAS)
)

print("\nColumnas esperadas que no aparecen:")
print(faltantes_columnas if faltantes_columnas else "Ninguna")

print("\nColumnas adicionales:")
print(columnas_no_esperadas if columnas_no_esperadas else "Ninguna")

if faltantes_columnas:
    raise ValueError(
        "El archivo no contiene todas las columnas esperadas. "
        "Revisa los nombres antes de continuar."
    )

Información general del DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 166732 entries, 0 to 166731
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype
---  ------                         --------------   -----
 0   Código Dane departamento       166732 non-null  str  
 1   Departamento                   166732 non-null  str  
 2   Código Dane municipio          166732 non-null  str  
 3   Municipio                      166732 non-null  str  
 4   Grupo cultivo                  166732 non-null  str  
 5   Subgrupo                       166732 non-null  str  
 6   Cultivo                        166732 non-null  str  
 7   Desagregación cultivo          166732 non-null  str  
 8   Año                            166732 non-null  str  
 9   Periodo                        166732 non-null  str  
 10  Área sembrada                  166732 non-null  str  
 11  Área cosechada                 166732 non-null  str  
 12  Producción                     166

### Interpretación de la estructura

En esta sección se comprueba que el archivo pueda leerse correctamente
y que contenga los atributos necesarios para el análisis.

Se debe registrar:

- El número total de registros y columnas.
- La presencia de todas las columnas esperadas.
- La existencia de columnas adicionales que deban investigarse.
- Los nombres y tipos iniciales de las variables.

La lectura inicial como texto es intencional y no significa que todas
las columnas deban permanecer en ese tipo de dato durante el proceso ETL.

In [9]:
# Considerar como vacíos los valores que contienen únicamente espacios.
df = df.replace(r"^\s*$", np.nan, regex=True)

resumen_nulos = pd.DataFrame({
    "columna": df.columns,
    "valores_nulos": df.isna().sum().values,
    "porcentaje_nulos": (
        df.isna().mean().values * 100
    ).round(4),
})

resumen_nulos = resumen_nulos.sort_values(
    "porcentaje_nulos",
    ascending=False,
)

print("Resumen de valores nulos y vacíos:")
display(resumen_nulos)

print("Total de celdas nulas:", int(df.isna().sum().sum()))
print("Filas con al menos un nulo:", int(df.isna().any(axis=1).sum()))

resumen_nulos.to_csv(
    OUTPUT_DIR / "valores_nulos.csv",
    index=False,
    encoding="utf-8-sig",
)

Resumen de valores nulos y vacíos:


,columna,valores_nulos,porcentaje_nulos
0,Código Dane departamento,0,0.0
1,Departamento,0,0.0
2,Código Dane municipio,0,0.0
3,Municipio,0,0.0
4,Grupo cultivo,0,0.0
5,Subgrupo,0,0.0
6,Cultivo,0,0.0
7,Desagregación cultivo,0,0.0
8,Año,0,0.0
9,Periodo,0,0.0


Total de celdas nulas: 0
Filas con al menos un nulo: 0


### Interpretación de valores nulos

Los valores nulos pueden afectar los cálculos, las relaciones entre tablas
y las reglas de validación posteriores.

La revisión permite determinar si existen campos incompletos y cuáles
son las columnas más afectadas.

Si no se identifican valores nulos, se documentará como una característica
observada de la fuente analizada. Sin embargo, esta condición deberá
verificarse nuevamente en cada ejecución del pipeline, porque futuras
actualizaciones podrían introducir valores faltantes.

No se imputarán valores durante esta etapa: cualquier tratamiento deberá
justificarse según el significado de la variable.

In [10]:
duplicados_exactos = df.duplicated(keep=False)

print(
    "Filas que forman parte de grupos de duplicados exactos:",
    int(duplicados_exactos.sum()),
)

print(
    "Cantidad de filas duplicadas después de la primera aparición:",
    int(df.duplicated().sum()),
)

if duplicados_exactos.any():
    display(
        df.loc[duplicados_exactos]
        .sort_values(list(df.columns))
        .head(20)
    )
else:
    print("No se identificaron duplicados exactos.")

resumen_duplicados = pd.DataFrame([{
    "total_registros": len(df),
    "filas_en_grupos_duplicados_exactos": int(duplicados_exactos.sum()),
    "duplicados_excluyendo_primera_aparicion": int(df.duplicated().sum()),
}])

resumen_duplicados.to_csv(
    OUTPUT_DIR / "duplicados_exactos.csv",
    index=False,
    encoding="utf-8-sig",
)

Filas que forman parte de grupos de duplicados exactos: 0
Cantidad de filas duplicadas después de la primera aparición: 0
No se identificaron duplicados exactos.


### Interpretación de duplicados exactos

Se consideran duplicados exactos los registros cuyos valores coinciden
en todas las columnas.

Esta revisión es distinta del análisis de granularidad: dos registros
pueden representar la misma combinación de municipio, cultivo y período
sin ser idénticos en todas sus columnas.

Por tanto, la ausencia de duplicados exactos no demuestra por sí sola
que cada observación tenga una clave única. Esa hipótesis se examinará
más adelante mediante claves candidatas.

## 4. Perfilamiento de las métricas agrícolas

Las variables Área sembrada, Área cosechada, Producción y Rendimiento
son fundamentales para analizar el comportamiento de la actividad agrícola.

Antes de calcular estadísticas, es necesario convertir sus valores a
formato numérico de forma controlada.

La fuente puede utilizar coma decimal y, dependiendo de la representación
de los valores, puntos como separadores de miles. Una conversión incorrecta
podría alterar significativamente las estadísticas y las conclusiones.

Por esta razón, se examinarán los formatos originales y se registrarán
los valores que no puedan convertirse.

In [11]:
for columna in COLUMNAS_METRICAS:
    print(f"\n--- {columna} ---")

    valores = (
        df[columna]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(15)
        .tolist()
    )

    print(valores)


--- Área sembrada ---
['8,00', '6,00', '7,00', '5,50', '0,16', '0,40', '0,50', '15,00', '12,00', '7,20', '465,00', '450,00', '437,58', '420,87', '368,73']

--- Área cosechada ---
['8,00', '7,00', '5,70', '6,60', '5,50', '0,16', '0,20', '0,30', '0,50', '15,00', '12,00', '6,00', '6,50', '6,70', '7,40']

--- Producción ---
['128,00', '152,00', '127,44', '133,00', '108,30', '125,40', '104,50', '0,30', '0,38', '0,27', '0,44', '105,00', '84,00', '56,00', '49,00']

--- Rendimiento ---
['16,00', '19,00', '15,93', '1,88', '1,90', '0,90', '0,88', '7,00', '8,00', '11,00', '1,24', '1,15', '0,73', '0,62', '0,71']


In [12]:
def convertir_numerico(serie: pd.Series) -> pd.Series:
    """
    Convierte una serie de texto a números.

    Reglas:
    - "1.234,56" -> 1234.56
    - "123,45"   -> 123.45
    - "1234.56"  -> 1234.56
    - "1234"     -> 1234

    Los valores no interpretables quedan como NaN.
    """

    valores = serie.astype("string").str.strip()

    # Eliminar espacios interiores, si se usan como separadores.
    valores = valores.str.replace(" ", "", regex=False)

    ambos = (
        valores.str.contains(",", na=False)
        & valores.str.contains(".", regex=False, na=False)
    )

    solo_coma = (
        valores.str.contains(",", na=False)
        & ~valores.str.contains(".", regex=False, na=False)
    )

    # Formato con punto de miles y coma decimal.
    valores.loc[ambos] = (
        valores.loc[ambos]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    # Formato con coma decimal.
    valores.loc[solo_coma] = (
        valores.loc[solo_coma]
        .str.replace(",", ".", regex=False)
    )

    return pd.to_numeric(valores, errors="coerce")

In [13]:
metricas_numericas = pd.DataFrame(index=df.index)
errores_conversion = []

for columna in COLUMNAS_METRICAS:
    original = df[columna]
    convertido = convertir_numerico(original)

    # Valores informados originalmente que no pudieron convertirse.
    error = original.notna() & convertido.isna()

    errores_conversion.append({
        "columna": columna,
        "valores_originales_no_nulos": int(original.notna().sum()),
        "valores_no_convertibles": int(error.sum()),
        "porcentaje_no_convertible": round(
            error.sum() / max(original.notna().sum(), 1) * 100,
            4,
        ),
    })

    metricas_numericas[columna] = convertido

resumen_conversion = pd.DataFrame(errores_conversion)

display(resumen_conversion)

resumen_conversion.to_csv(
    OUTPUT_DIR / "errores_conversion_numerica.csv",
    index=False,
    encoding="utf-8-sig",
)

display(metricas_numericas.head())

,columna,valores_originales_no_nulos,valores_no_convertibles,porcentaje_no_convertible
0,Área sembrada,166732,0,0.0
1,Área cosechada,166732,0,0.0
2,Producción,166732,0,0.0
3,Rendimiento,166732,0,0.0


,Área sembrada,Área cosechada,Producción,Rendimiento
0,8.0,8.0,128.0,16.0
1,8.0,8.0,152.0,19.0
2,8.0,8.0,127.44,15.93
3,8.0,8.0,152.0,19.0
4,8.0,7.0,133.0,19.0


In [14]:
estadisticas = []

for columna in COLUMNAS_METRICAS:
    serie = metricas_numericas[columna]

    estadisticas.append({
        "columna": columna,
        "registros_validos": int(serie.notna().sum()),
        "valores_nulos_o_no_convertibles": int(serie.isna().sum()),
        "minimo": serie.min(),
        "maximo": serie.max(),
        "media": serie.mean(),
        "mediana": serie.median(),
        "desviacion_estandar": serie.std(),
        "percentil_1": serie.quantile(0.01),
        "cuartil_1": serie.quantile(0.25),
        "cuartil_3": serie.quantile(0.75),
        "percentil_99": serie.quantile(0.99),
        "ceros": int((serie == 0).sum()),
        "negativos": int((serie < 0).sum()),
    })

estadisticas_df = pd.DataFrame(estadisticas)

display(estadisticas_df.round(4))

estadisticas_df.to_csv(
    OUTPUT_DIR / "metricas_estadisticas.csv",
    index=False,
    encoding="utf-8-sig",
)

,columna,registros_validos,valores_nulos_o_no_convertibles,minimo,maximo,media,mediana,desviacion_estandar,percentil_1,cuartil_1,cuartil_3,percentil_99,ceros,negativos
0,Área sembrada,166732,0,0.0,77630.0,228.4685,20.0,1192.1304,0.0,5.0,90.0,3724.830,3007,0
1,Área cosechada,166732,0,0.0,77630.0,208.9916,17.0,1120.0537,0.0,5.0,80.0,3424.659,7116,0
2,Producción,166732,0,0.0,4776340.5,3189.6355,100.0,49246.1670,0.0,20.0,549.0,38286.900,7116,0
3,Rendimiento,166732,0,0.0,253.0,10.6012,6.0,15.5740,0.0,1.8,13.0,85.000,7116,0


### Interpretación de las métricas

Las estadísticas permiten conocer la escala, la dispersión y los extremos
de cada variable agrícola.

- **Mínimo y máximo:** muestran el rango observado.
- **Media y mediana:** permiten comparar el valor promedio con el central
  y detectar posibles distribuciones asimétricas.
- **Desviación estándar:** aproxima la dispersión de los valores.
- **Percentiles:** ayudan a examinar extremos sin depender exclusivamente
  del mínimo y el máximo.
- **Ceros y negativos:** permiten identificar valores que requieren
  interpretación según la definición de cada variable.
- **Valores no convertibles:** evidencian problemas de formato o valores
  que requieren revisión.

Un valor extremo no se clasificará automáticamente como error. La
producción agrícola puede variar considerablemente según el cultivo,
el municipio, la unidad de medida y el período observado.

Antes de utilizar estas métricas en el DW, se debe confirmar que la
conversión numérica represente correctamente los valores originales.

## 5. Análisis temporal

El dataset contiene los atributos Año y Periodo. Su análisis permite
identificar los períodos disponibles, revisar la consistencia entre ambos
campos y detectar registros con formatos temporales inesperados.

Es importante distinguir los registros anuales de los semestrales.
Si la fuente contiene observaciones identificadas, por ejemplo, como
2021A y 2021B, estas deben conservarse como períodos diferenciados.

No se asumirá que un registro anual equivale a la suma de los semestres,
ni se sumarán registros anuales y semestrales como si fueran observaciones
independientes de una misma granularidad.

In [15]:
df_temporal = df[["Año", "Periodo"]].copy()

df_temporal["Año"] = df_temporal["Año"].str.strip()
df_temporal["Periodo"] = df_temporal["Periodo"].str.strip()

# Extraer el año al inicio del período: 2021A -> 2021.
df_temporal["anio_extraido"] = (
    df_temporal["Periodo"]
    .str.extract(r"^(\d{4})", expand=False)
)

df_temporal["anio_extraido"] = pd.to_numeric(
    df_temporal["anio_extraido"],
    errors="coerce",
).astype("Int64")

df_temporal["anio_columna"] = pd.to_numeric(
    df_temporal["Año"],
    errors="coerce",
).astype("Int64")

print("Distribución de Año:")
display(
    df_temporal["Año"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Año")
    .reset_index(name="registros")
)

print("Distribución de Periodo:")
display(
    df_temporal["Periodo"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Periodo")
    .reset_index(name="registros")
)

print("Distribución conjunta Año y Periodo:")
display(
    df_temporal
    .groupby(["Año", "Periodo"], dropna=False)
    .size()
    .reset_index(name="registros")
    .sort_values(["Año", "Periodo"], na_position="last")
)

Distribución de Año:


,Año,registros
0,2019,20436
1,2020,21511
2,2021,23898
3,2022,24808
4,2023,24909
5,2024,25553
6,2025,25617


Distribución de Periodo:


,Periodo,registros
0,2019,8007
1,2019A,6753
2,2019B,5676
3,2020,8572
4,2020A,6923
5,2020B,6016
6,2021,9527
7,2021A,7654
8,2021B,6717
9,2022,9627


Distribución conjunta Año y Periodo:


,Año,Periodo,registros
0,2019,2019,8007
1,2019,2019A,6753
2,2019,2019B,5676
3,2020,2020,8572
4,2020,2020A,6923
5,2020,2020B,6016
6,2021,2021,9527
7,2021,2021A,7654
8,2021,2021B,6717
9,2022,2022,9627


In [16]:
# Detectar períodos que no siguen los formatos esperados:
# YYYY, YYYYA o YYYYB.
formato_periodo_valido = df_temporal["Periodo"].str.fullmatch(
    r"\d{4}[AB]?",
    na=False,
)

# Detectar años que no pueden convertirse a entero.
anio_columna_valido = df_temporal["anio_columna"].notna()

# Detectar períodos cuyo año inicial no coincide con Año.
anio_coincide = (
    df_temporal["anio_columna"]
    == df_temporal["anio_extraido"]
)

problemas_temporales = df_temporal.loc[
    (~formato_periodo_valido)
    | (~anio_columna_valido)
    | (df_temporal["anio_extraido"].isna())
    | (~anio_coincide.fillna(False))
].copy()

print("Registros con problemas temporales:", len(problemas_temporales))
display(problemas_temporales.head(20))

problemas_temporales.to_csv(
    OUTPUT_DIR / "problemas_temporales.csv",
    index=False,
    encoding="utf-8-sig",
)

# Resumen de coincidencia entre Año y el año extraído de Periodo.
resumen_temporal = pd.DataFrame([{
    "total_registros": len(df_temporal),
    "periodos_con_formato_no_esperado": int((~formato_periodo_valido).sum()),
    "años_no_convertibles": int((~anio_columna_valido).sum()),
    "años_extraidos_no_convertibles": int(
        df_temporal["anio_extraido"].isna().sum()
    ),
    "años_no_coincidentes": int(
        (~anio_coincide.fillna(False)).sum()
    ),
}])

display(resumen_temporal)

resumen_temporal.to_csv(
    OUTPUT_DIR / "resumen_temporal.csv",
    index=False,
    encoding="utf-8-sig",
)

Registros con problemas temporales: 0


,Año,Periodo,anio_extraido,anio_columna


,total_registros,periodos_con_formato_no_esperado,años_no_convertibles,años_extraidos_no_convertibles,años_no_coincidentes
0,166732,0,0,0,0


### Interpretación temporal

Se deben revisar la distribución de años, la frecuencia de cada período
y los registros identificados como inconsistentes.

Un formato inesperado no demuestra necesariamente que el dato sea
incorrecto: puede existir una convención de publicación adicional que
deba consultarse en la documentación oficial.

Para el modelo dimensional, Año y Periodo deben representar la
observación temporal de origen sin mezclar diferentes niveles de
agregación.

Antes de integrar la fuente climática, se deberá establecer cómo se
interpretan los períodos anuales y semestrales y cómo se agregan las
variables climáticas diarias a cada uno de ellos.

## 6. Análisis geográfico

El análisis geográfico busca comprobar la consistencia de los códigos
y nombres utilizados para identificar departamentos y municipios.

Los códigos DANE deben conservarse como texto. Su función es identificar
entidades territoriales, no expresar cantidades.

También se examinarán las relaciones entre códigos y nombres para
identificar casos en los que un mismo código aparezca asociado a varios
nombres, o un mismo nombre esté relacionado con distintos códigos.

Estas relaciones se investigarán antes de utilizarlas como base de una
dimensión geográfica o para relacionar la fuente agrícola con la tabla
de coordenadas municipales que se empleará en la extracción climática.

In [17]:
df_geo = df[COLUMNAS_GEOGRAFICAS].copy()

for columna in COLUMNAS_GEOGRAFICAS:
    df_geo[columna] = df_geo[columna].str.strip()

COLUMNAS_CODIGOS = [
    "Código Dane departamento",
    "Código Dane municipio",
]

for columna in COLUMNAS_CODIGOS:
    df_geo[columna] = df_geo[columna].replace("", np.nan)

print("Ejemplos de códigos y nombres geográficos:")
display(df_geo.head(10))

print("\nLongitudes de los códigos de departamento:")
display(
    df_geo["Código Dane departamento"]
    .dropna()
    .str.len()
    .value_counts()
    .sort_index()
    .rename_axis("longitud")
    .reset_index(name="registros")
)

print("\nLongitudes de los códigos de municipio:")
display(
    df_geo["Código Dane municipio"]
    .dropna()
    .str.len()
    .value_counts()
    .sort_index()
    .rename_axis("longitud")
    .reset_index(name="registros")
)

Ejemplos de códigos y nombres geográficos:


,Código Dane departamento,Departamento,Código Dane municipio,Municipio
0,05,Antioquia,05001,Medellín
1,05,Antioquia,05001,Medellín
2,05,Antioquia,05001,Medellín
3,05,Antioquia,05001,Medellín
4,05,Antioquia,05001,Medellín
5,05,Antioquia,05001,Medellín
6,05,Antioquia,05001,Medellín
7,05,Antioquia,05001,Medellín
8,05,Antioquia,05001,Medellín
9,05,Antioquia,05001,Medellín



Longitudes de los códigos de departamento:


,longitud,registros
0,2,166732



Longitudes de los códigos de municipio:


,longitud,registros
0,5,166732


In [18]:
def analizar_relacion(
    data: pd.DataFrame,
    columna_codigo: str,
    columna_nombre: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Devuelve:
    1. Códigos asociados con más de un nombre.
    2. Nombres asociados con más de un código.
    """

    pares = data[
        [columna_codigo, columna_nombre]
    ].dropna().drop_duplicates()

    nombres_por_codigo = (
        pares.groupby(columna_codigo)[columna_nombre]
        .nunique()
    )

    codigos_por_nombre = (
        pares.groupby(columna_nombre)[columna_codigo]
        .nunique()
    )

    codigos_inconsistentes = nombres_por_codigo[
        nombres_por_codigo > 1
    ].index

    nombres_inconsistentes = codigos_por_nombre[
        codigos_por_nombre > 1
    ].index

    detalle_codigo = pares[
        pares[columna_codigo].isin(codigos_inconsistentes)
    ].sort_values([columna_codigo, columna_nombre])

    detalle_nombre = pares[
        pares[columna_nombre].isin(nombres_inconsistentes)
    ].sort_values([columna_nombre, columna_codigo])

    return detalle_codigo, detalle_nombre


problemas_depto_codigo, problemas_depto_nombre = analizar_relacion(
    df_geo,
    "Código Dane departamento",
    "Departamento",
)

problemas_municipio_codigo, problemas_municipio_nombre = analizar_relacion(
    df_geo,
    "Código Dane municipio",
    "Municipio",
)

print("Departamentos: códigos asociados con múltiples nombres")
display(problemas_depto_codigo)

print("Departamentos: nombres asociados con múltiples códigos")
display(problemas_depto_nombre)

print("Municipios: códigos asociados con múltiples nombres")
display(problemas_municipio_codigo.head(50))

print("Municipios: nombres asociados con múltiples códigos")
display(problemas_municipio_nombre.head(50))

problemas_depto_codigo.to_csv(
    OUTPUT_DIR / "departamentos_codigo_multiples_nombres.csv",
    index=False,
    encoding="utf-8-sig",
)

problemas_depto_nombre.to_csv(
    OUTPUT_DIR / "departamentos_nombre_multiples_codigos.csv",
    index=False,
    encoding="utf-8-sig",
)

problemas_municipio_codigo.to_csv(
    OUTPUT_DIR / "municipios_codigo_multiples_nombres.csv",
    index=False,
    encoding="utf-8-sig",
)

problemas_municipio_nombre.to_csv(
    OUTPUT_DIR / "municipios_nombre_multiples_codigos.csv",
    index=False,
    encoding="utf-8-sig",
)

Departamentos: códigos asociados con múltiples nombres


,Código Dane departamento,Departamento


Departamentos: nombres asociados con múltiples códigos


,Código Dane departamento,Departamento


Municipios: códigos asociados con múltiples nombres


,Código Dane municipio,Municipio


Municipios: nombres asociados con múltiples códigos


,Código Dane municipio,Municipio
44551,18029,Albania
93014,44035,Albania
126618,68020,Albania
62163,25019,Albán
103926,52019,Albán
1875,05055,Argelia
46458,19050,Argelia
150544,76054,Argelia
1973,05059,Armenia
122420,63001,Armenia


In [19]:
pares_geo = (
    df_geo[
        [
            "Código Dane departamento",
            "Departamento",
            "Código Dane municipio",
            "Municipio",
        ]
    ]
    .dropna()
    .drop_duplicates()
)

# Cada código municipal debería estar relacionado con un único
# departamento y un único nombre municipal en esta fuente.
municipios_por_codigo = (
    pares_geo.groupby("Código Dane municipio")
    .agg(
        cantidad_departamentos=(
            "Código Dane departamento", "nunique"
        ),
        cantidad_nombres=(
            "Municipio", "nunique"
        ),
    )
    .reset_index()
)

municipios_sospechosos = municipios_por_codigo[
    (municipios_por_codigo["cantidad_departamentos"] > 1)
    | (municipios_por_codigo["cantidad_nombres"] > 1)
]

print(
    "Códigos municipales asociados con múltiples departamentos "
    "o nombres:"
)
display(municipios_sospechosos)

municipios_sospechosos.to_csv(
    OUTPUT_DIR / "municipios_relaciones_inconsistentes.csv",
    index=False,
    encoding="utf-8-sig",
)

Códigos municipales asociados con múltiples departamentos o nombres:


,Código Dane municipio,cantidad_departamentos,cantidad_nombres


### Interpretación geográfica

Los códigos DANE constituyen el punto de partida para relacionar la
fuente agrícola con el catálogo de municipios y sus coordenadas.

Se debe revisar cualquier código asociado a múltiples nombres, cualquier
nombre vinculado a varios códigos y cualquier municipio asociado a más
de un departamento.

Los casos encontrados deben investigarse antes de calificarlos como
errores: pueden reflejar diferencias de escritura, cambios en la
referencia territorial, particularidades de la fuente o inconsistencias
reales.

Para la integración con Open-Meteo, la relación prevista es:

Código DANE del municipio
→ código del catálogo geográfico
→ coordenadas representativas del municipio
→ extracción de variables climáticas.

Las coordenadas municipales representan una ubicación de referencia
y no necesariamente las condiciones climáticas de cada parcela agrícola.

## 7. Análisis de los atributos agrícolas

Los atributos agrícolas describen la clasificación de los cultivos y
permiten estudiar las relaciones entre grupos, subgrupos, cultivos,
desagregaciones, códigos y nombres científicos.

El objetivo es conocer los valores existentes, identificar combinaciones
observadas y detectar relaciones que requieran revisión.

Una relación de uno a varios no se considerará automáticamente un error.
Su validez depende de cómo la fuente define y utiliza cada atributo.

In [20]:
for columna in COLUMNAS_AGRICOLAS:
    print(f"\n--- Distribución de {columna} ---")

    distribucion = (
        df[columna]
        .fillna("[NULO]")
        .value_counts(dropna=False)
        .rename_axis(columna)
        .reset_index(name="registros")
    )

    print("Valores distintos:", df[columna].nunique(dropna=True))
    display(distribucion.head(20))

    nombre_archivo = (
        columna.lower()
        .replace(" ", "_")
        .replace("ó", "o")
        .replace("í", "i")
        .replace("á", "a")
        .replace("é", "e")
        .replace("ú", "u")
    )

    distribucion.to_csv(
        OUTPUT_DIR / f"distribucion_{nombre_archivo}.csv",
        index=False,
        encoding="utf-8-sig",
    )


--- Distribución de Grupo cultivo ---
Valores distintos: 8


,Grupo cultivo,registros
0,Frutales,49152
1,Hortalizas,31629
2,Cereales,28663
3,Raíces y tubérculos,21244
4,Leguminosas,17898
5,Cultivos tropicales tradicionales,14511
6,Oleaginosas,2045
7,"Cultivos para condimentos, bebidas medicinales...",1590



--- Distribución de Subgrupo ---
Valores distintos: 23


,Subgrupo,registros
0,Cereales,28663
1,Demás frutales,23848
2,Raíces y tubérculos,21244
3,Hortalizas de fruto,19082
4,Leguminosas,17898
5,Cultivos tropicales tradicionales,14511
6,Cítricos,11124
7,Solanáceas,4707
8,Pasifloráceas,4244
9,Hortalizas de hoja,4023



--- Distribución de Cultivo ---
Valores distintos: 165


...

,Desagregación cultivo,registros
0,Yuca consumo en fresco,9677
1,Frijol,9642
2,Maíz amarillo tradicional,9247
3,Plátano consumo interno,5600
4,Maíz blanco tradicional,4928
5,Café,4459
6,Arveja,4408
7,Tomate,4350
8,Cacao,4034
9,Caña panelera,3951



--- Distribución de Código del cultivo ---
Valores distintos: 194


,Código del cultivo,registros
0,1081001,9677
1,1060200,9642
2,1010402,9247
3,2045802,5600
4,1010404,4928
5,2030300,4459
6,1060100,4408
7,1052901,4350
8,2030200,4034
9,2030402,3951



--- Distribución de Nombre científico del cultivo ---
Valores distintos: 158


,Nombre científico del cultivo,registros
0,Zea mays,21992
1,Manihot esculenta,10126
2,Phaseolus vulgaris,9642
3,Lycopersicon esculentum,7174
4,Musa x paradisiaca,5652
5,Persea americana,5618
6,Citrus limon,4839
7,Oryza sativa,4838
8,Saccharum officinarum,4702
9,Coffea arabica,4459


In [21]:
def resumir_relacion(
    data: pd.DataFrame,
    columna_origen: str,
    columna_destino: str,
) -> pd.DataFrame:
    pares = (
        data[[columna_origen, columna_destino]]
        .dropna()
        .drop_duplicates()
    )

    resumen = (
        pares.groupby(columna_origen)[columna_destino]
        .agg(
            cantidad_valores_distintos="nunique",
        )
        .reset_index()
    )

    resumen = resumen.sort_values(
        "cantidad_valores_distintos",
        ascending=False,
    )

    return resumen


relaciones_agricolas = [
    ("Grupo cultivo", "Subgrupo"),
    ("Subgrupo", "Cultivo"),
    ("Cultivo", "Desagregación cultivo"),
    ("Cultivo", "Código del cultivo"),
    ("Código del cultivo", "Cultivo"),
    ("Código del cultivo", "Nombre científico del cultivo"),
]

for origen, destino in relaciones_agricolas:
    print(f"\nRelación: {origen} → {destino}")

    resumen = resumir_relacion(df, origen, destino)

    display(resumen.head(20))

    # Número de valores de origen con más de un valor asociado.
    multiples = resumen[
        resumen["cantidad_valores_distintos"] > 1
    ]

    print(
        f"Valores de '{origen}' asociados con varios valores "
        f"de '{destino}': {len(multiples):,}"
    )

    archivo = (
        f"relacion_{origen.lower().replace(' ', '_')}"
        f"__{destino.lower().replace(' ', '_')}.csv"
    )

    resumen.to_csv(
        OUTPUT_DIR / archivo,
        index=False,
        encoding="utf-8-sig",
    )


Relación: Grupo cultivo → Subgrupo


,Grupo cultivo,cantidad_valores_distintos
3,Frutales,8
4,Hortalizas,6
1,"Cultivos para condimentos, bebidas medicinales...",4
0,Cereales,1
2,Cultivos tropicales tradicionales,1
5,Leguminosas,1
6,Oleaginosas,1
7,Raíces y tubérculos,1


Valores de 'Grupo cultivo' asociados con varios valores de 'Subgrupo': 3

Relación: Subgrupo → Cultivo


,Subgrupo,cantidad_valores_distintos
9,Demás frutales,33
12,Hortalizas de fruto,11
21,Raíces y tubérculos,11
8,Cítricos,9
5,Condimentos,9
4,Cereales,8
16,Leguminosas,8
7,Cultivos tropicales tradicionales,8
1,Aromáticas,7
17,Medicinales,7


Valores de 'Subgrupo' asociados con varios valores de 'Cultivo': 22

Relación: Cultivo → Desagregación cultivo


...

Valores de 'Cultivo' asociados con varios valores de 'Código del cultivo': 16

Relación: Código del cultivo → Cultivo


,Código del cultivo,cantidad_valores_distintos
0,1010101,1
1,1010102,1
2,1010103,1
3,1010200,1
4,1010300,1
5,1010401,1
6,1010402,1
7,1010403,1
8,1010404,1
9,1010497,1


Valores de 'Código del cultivo' asociados con varios valores de 'Cultivo': 0

Relación: Código del cultivo → Nombre científico del cultivo


,Código del cultivo,cantidad_valores_distintos
0,1010101,1
1,1010102,1
2,1010103,1
3,1010200,1
4,1010300,1
5,1010401,1
6,1010402,1
7,1010403,1
8,1010404,1
9,1010497,1


Valores de 'Código del cultivo' asociados con varios valores de 'Nombre científico del cultivo': 0


In [22]:
def resumir_relacion(
    data: pd.DataFrame,
    columna_origen: str,
    columna_destino: str,
) -> pd.DataFrame:
    pares = (
        data[[columna_origen, columna_destino]]
        .dropna()
        .drop_duplicates()
    )

    resumen = (
        pares.groupby(columna_origen)[columna_destino]
        .agg(
            cantidad_valores_distintos="nunique",
        )
        .reset_index()
    )

    resumen = resumen.sort_values(
        "cantidad_valores_distintos",
        ascending=False,
    )

    return resumen


relaciones_agricolas = [
    ("Grupo cultivo", "Subgrupo"),
    ("Subgrupo", "Cultivo"),
    ("Cultivo", "Desagregación cultivo"),
    ("Cultivo", "Código del cultivo"),
    ("Código del cultivo", "Cultivo"),
    ("Código del cultivo", "Nombre científico del cultivo"),
]

for origen, destino in relaciones_agricolas:
    print(f"\nRelación: {origen} → {destino}")

    resumen = resumir_relacion(df, origen, destino)

    display(resumen.head(20))

    # Número de valores de origen con más de un valor asociado.
    multiples = resumen[
        resumen["cantidad_valores_distintos"] > 1
    ]

    print(
        f"Valores de '{origen}' asociados con varios valores "
        f"de '{destino}': {len(multiples):,}"
    )

    archivo = (
        f"relacion_{origen.lower().replace(' ', '_')}"
        f"__{destino.lower().replace(' ', '_')}.csv"
    )

    resumen.to_csv(
        OUTPUT_DIR / archivo,
        index=False,
        encoding="utf-8-sig",
    )


Relación: Grupo cultivo → Subgrupo


,Grupo cultivo,cantidad_valores_distintos
3,Frutales,8
4,Hortalizas,6
1,"Cultivos para condimentos, bebidas medicinales...",4
0,Cereales,1
2,Cultivos tropicales tradicionales,1
5,Leguminosas,1
6,Oleaginosas,1
7,Raíces y tubérculos,1


Valores de 'Grupo cultivo' asociados con varios valores de 'Subgrupo': 3

Relación: Subgrupo → Cultivo


,Subgrupo,cantidad_valores_distintos
9,Demás frutales,33
12,Hortalizas de fruto,11
21,Raíces y tubérculos,11
8,Cítricos,9
5,Condimentos,9
4,Cereales,8
16,Leguminosas,8
7,Cultivos tropicales tradicionales,8
1,Aromáticas,7
17,Medicinales,7


Valores de 'Subgrupo' asociados con varios valores de 'Cultivo': 22

Relación: Cultivo → Desagregación cultivo


...

Valores de 'Cultivo' asociados con varios valores de 'Código del cultivo': 16

Relación: Código del cultivo → Cultivo


,Código del cultivo,cantidad_valores_distintos
0,1010101,1
1,1010102,1
2,1010103,1
3,1010200,1
4,1010300,1
5,1010401,1
6,1010402,1
7,1010403,1
8,1010404,1
9,1010497,1


Valores de 'Código del cultivo' asociados con varios valores de 'Cultivo': 0

Relación: Código del cultivo → Nombre científico del cultivo


,Código del cultivo,cantidad_valores_distintos
0,1010101,1
1,1010102,1
2,1010103,1
3,1010200,1
4,1010300,1
5,1010401,1
6,1010402,1
7,1010403,1
8,1010404,1
9,1010497,1


Valores de 'Código del cultivo' asociados con varios valores de 'Nombre científico del cultivo': 0


### Interpretación de las relaciones agrícolas

Las distribuciones permiten identificar los principales grupos, subgrupos
y cultivos presentes en el dataset.

El análisis de relaciones ayuda a determinar si un atributo puede
utilizarse como identificador de otro o si se necesita conservar una
combinación de atributos.

Por ejemplo, si un nombre de cultivo está asociado con varios códigos,
no se debe asumir que el nombre por sí solo identifica de manera única
el cultivo en la fuente.

Las relaciones observadas deben contrastarse con la definición oficial
de los códigos agrícolas antes de establecer claves definitivas para
la dimensión de cultivos.

Los resultados también permiten identificar atributos potencialmente
redundantes y orientar el diseño de las reglas de calidad.

In [23]:
for columna in COLUMNAS_CATEGORICAS:
    resumen = (
        df[columna]
        .fillna("[NULO]")
        .value_counts(dropna=False)
        .rename_axis(columna)
        .reset_index(name="registros")
    )

    print(f"\nDistribución de {columna}:")
    display(resumen)

    resumen.to_csv(
        OUTPUT_DIR
        / f"distribucion_{columna.lower().replace(' ', '_')}.csv",
        index=False,
        encoding="utf-8-sig",
    )


Distribución de Ciclo del cultivo:


,Ciclo del cultivo,registros
0,Transitorio,101676
1,Permanente,65056



Distribución de Estado físico del cultivo:


,Estado físico del cultivo,registros
0,En fresco,113470
1,Grano seco,32403
2,Paddy o cascara verde,4838
3,Caña o verde,4702
4,Pergamino o seco de trilla,4459
5,Grano,4408
6,Aceite crudo,976
7,Grano o almendra,681
8,Fibra o cabuya,433
9,Algodón semilla,362


### Interpretación del ciclo y estado físico

La distribución de estas variables permite conocer las categorías
que utiliza la fuente para caracterizar los cultivos.

Se revisarán categorías inesperadas, valores nulos y diferencias
de escritura que puedan generar agrupaciones incorrectas.

No se crearán categorías nuevas ni se unificarán valores únicamente
por similitud textual. Cualquier normalización debe apoyarse en las
definiciones de la fuente y conservar la trazabilidad del valor original.

## 8. Evaluación de la granularidad

La granularidad representa el nivel de detalle que describe cada fila
de una tabla. En esta fuente se busca determinar qué combinación de
atributos identifica una observación agrícola.

Se evaluarán dos claves candidatas:

1. Código Dane municipio + Código del cultivo + Año + Periodo.
2. La misma combinación, incluyendo Desagregación cultivo.

El análisis permitirá comprobar cuántos registros comparten la misma
combinación de valores.

La clave definitiva no se establecerá solamente porque no haya
duplicados. También debe representar correctamente el significado
de una observación y ser compatible con las reglas del negocio.

In [24]:
CLAVE_CANDIDATA_1 = [
    "Código Dane municipio",
    "Código del cultivo",
    "Año",
    "Periodo",
]

CLAVE_CANDIDATA_2 = [
    "Código Dane municipio",
    "Código del cultivo",
    "Desagregación cultivo",
    "Año",
    "Periodo",
]


def evaluar_granularidad(
    data: pd.DataFrame,
    columnas_clave: list[str],
) -> tuple[dict, pd.DataFrame]:
    """
    Evalúa una clave candidata e identifica las combinaciones repetidas.
    Los nulos se conservan como grupo para no ocultar problemas.
    """

    grupos = (
        data.groupby(
            columnas_clave,
            dropna=False,
        )
        .size()
        .reset_index(name="cantidad_registros")
    )

    grupos_repetidos = grupos[
        grupos["cantidad_registros"] > 1
    ].copy()

    total_grupos = len(grupos)
    grupos_duplicados = len(grupos_repetidos)
    filas_en_grupos_duplicados = int(
        grupos_repetidos["cantidad_registros"].sum()
    )

    resumen = {
        "clave_evaluada": " + ".join(columnas_clave),
        "registros_totales": len(data),
        "combinaciones_distintas": total_grupos,
        "combinaciones_repetidas": grupos_duplicados,
        "filas_en_combinaciones_repetidas": filas_en_grupos_duplicados,
        "filas_que_exceden_una_por_grupo": int(
            grupos_repetidos["cantidad_registros"].sub(1).sum()
        ),
        "porcentaje_combinaciones_repetidas": round(
            grupos_duplicados / max(total_grupos, 1) * 100,
            4,
        ),
    }

    return resumen, grupos_repetidos


resumen_clave_1, repetidos_clave_1 = evaluar_granularidad(
    df,
    CLAVE_CANDIDATA_1,
)

resumen_clave_2, repetidos_clave_2 = evaluar_granularidad(
    df,
    CLAVE_CANDIDATA_2,
)

resumen_granularidad = pd.DataFrame([
    resumen_clave_1,
    resumen_clave_2,
])

display(resumen_granularidad)

resumen_granularidad.to_csv(
    OUTPUT_DIR / "evaluacion_granularidad.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Repeticiones de la primera clave candidata:")
display(repetidos_clave_1.head(30))

print("Repeticiones de la segunda clave candidata:")
display(repetidos_clave_2.head(30))

repetidos_clave_1.to_csv(
    OUTPUT_DIR / "repetidos_clave_candidata_1.csv",
    index=False,
    encoding="utf-8-sig",
)

repetidos_clave_2.to_csv(
    OUTPUT_DIR / "repetidos_clave_candidata_2.csv",
    index=False,
    encoding="utf-8-sig",
)

,clave_evaluada,registros_totales,combinaciones_distintas,combinaciones_repetidas,filas_en_combinaciones_repetidas,filas_que_exceden_una_por_grupo,porcentaje_combinaciones_repetidas
0,Código Dane municipio + Código del cultivo + A...,166732,166732,0,0,0,0.0
1,Código Dane municipio + Código del cultivo + D...,166732,166732,0,0,0,0.0


Repeticiones de la primera clave candidata:


,Código Dane municipio,Código del cultivo,Año,Periodo,cantidad_registros


Repeticiones de la segunda clave candidata:


,Código Dane municipio,Código del cultivo,Desagregación cultivo,Año,Periodo,cantidad_registros


In [25]:
for nombre, clave in [
    ("Clave candidata 1", CLAVE_CANDIDATA_1),
    ("Clave candidata 2", CLAVE_CANDIDATA_2),
]:
    mascara_incompleta = df[clave].isna().any(axis=1)

    print(f"\n{nombre}")
    print("Columnas:", clave)
    print(
        "Registros con al menos un componente nulo:",
        int(mascara_incompleta.sum()),
    )

    if mascara_incompleta.any():
        display(df.loc[mascara_incompleta, clave].head(20))


Clave candidata 1
Columnas: ['Código Dane municipio', 'Código del cultivo', 'Año', 'Periodo']
Registros con al menos un componente nulo: 0

Clave candidata 2
Columnas: ['Código Dane municipio', 'Código del cultivo', 'Desagregación cultivo', 'Año', 'Periodo']
Registros con al menos un componente nulo: 0


### Interpretación de la granularidad

La comparación entre claves candidatas permite evaluar si la inclusión
de Desagregación cultivo modifica la cantidad de combinaciones repetidas.

Si una clave presenta combinaciones repetidas, deben investigarse los
registros afectados antes de concluir que existe un error. Las repeticiones
pueden indicar que falta un atributo en la clave, que existen diferencias
legítimas en el nivel de detalle o que la granularidad inicialmente
propuesta debe revisarse.

También deben examinarse los componentes nulos, ya que una combinación
incompleta no ofrece la misma garantía de identificación que una clave
completa.

La decisión final deberá ser coherente con el significado de cada fila,
la documentación de EVA y el grano que se conservará en la tabla de hechos.

## 9. Consistencia entre las métricas agrícolas

Además del análisis estadístico, se examinarán algunas relaciones entre
las variables cuantitativas.

Se prestará atención a:

- Área cosechada superior al área sembrada.
- Área sembrada igual a cero y área cosechada positiva.
- Valores negativos en las métricas.
- Posibles relaciones entre producción, área cosechada y rendimiento.

Estos casos se consideran indicadores para investigar, no errores
automáticos. Por ejemplo, el área cosechada puede superar el área
sembrada en determinadas situaciones agrícolas debido a diferencias
en las definiciones, los ciclos o la metodología de reporte.

La interpretación final requiere conocer las unidades y definiciones
oficiales de las variables.

In [26]:
area_sembrada = metricas_numericas["Área sembrada"]
area_cosechada = metricas_numericas["Área cosechada"]
produccion = metricas_numericas["Producción"]
rendimiento = metricas_numericas["Rendimiento"]

casos_area_cosechada_mayor = (
    area_sembrada.notna()
    & area_cosechada.notna()
    & (area_cosechada > area_sembrada)
)

casos_sembrada_cero_cosechada_positiva = (
    (area_sembrada == 0)
    & (area_cosechada > 0)
)

casos_metricas_negativas = (
    metricas_numericas[COLUMNAS_METRICAS] < 0
).any(axis=1)

resumen_consistencia = pd.DataFrame([
    {
        "indicador": "Área cosechada > área sembrada",
        "registros": int(casos_area_cosechada_mayor.sum()),
        "porcentaje_total": round(
            casos_area_cosechada_mayor.mean() * 100, 4
        ),
    },
    {
        "indicador": "Área sembrada = 0 y área cosechada > 0",
        "registros": int(
            casos_sembrada_cero_cosechada_positiva.sum()
        ),
        "porcentaje_total": round(
            casos_sembrada_cero_cosechada_positiva.mean() * 100, 4
        ),
    },
    {
        "indicador": "Al menos una métrica negativa",
        "registros": int(casos_metricas_negativas.sum()),
        "porcentaje_total": round(
            casos_metricas_negativas.mean() * 100, 4
        ),
    },
])

display(resumen_consistencia)

resumen_consistencia.to_csv(
    OUTPUT_DIR / "consistencia_metricas.csv",
    index=False,
    encoding="utf-8-sig",
)

# Conservar las filas originales para investigar los casos.
casos_investigacion = (
    casos_area_cosechada_mayor
    | casos_sembrada_cero_cosechada_positiva
    | casos_metricas_negativas
)

columnas_revision = (
    COLUMNAS_GEOGRAFICAS
    + COLUMNAS_AGRICOLAS
    + COLUMNAS_TEMPORALES
    + COLUMNAS_METRICAS
)

df.loc[casos_investigacion, columnas_revision].to_csv(
    OUTPUT_DIR / "registros_para_investigar_metricas.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Registros identificados para investigación:",
    int(casos_investigacion.sum()),
)

,indicador,registros,porcentaje_total
0,Área cosechada > área sembrada,13208,7.9217
1,Área sembrada = 0 y área cosechada > 0,3003,1.8011
2,Al menos una métrica negativa,0,0.0000


Registros identificados para investigación: 13208


### Interpretación de la consistencia de métricas

Los resultados permiten dimensionar las situaciones que deberán
investigarse antes de establecer las reglas definitivas de calidad.

Un caso detectado por este análisis no se eliminará automáticamente.
Primero se deberá verificar si corresponde a una situación válida,
una diferencia metodológica, un problema de formato o una inconsistencia
real de los datos.

Las reglas productivas deberán diferenciar entre:

- **Error crítico:** impide aceptar o cargar registros cuando compromete
  la integridad o el significado de la información.
- **Advertencia:** identifica una situación que requiere seguimiento,
  pero que puede ser válida según el contexto.
- **Indicador descriptivo:** permite monitorear el comportamiento de la
  fuente sin determinar por sí solo su aceptación.

Esta clasificación se formalizará en el catálogo de reglas de calidad
y se implementará posteriormente mediante Great Expectations y la
orquestación del pipeline.

In [27]:
resumen_general = {
    "archivo_fuente": CSV_PATH.name,
    "total_registros": int(len(df)),
    "total_columnas": int(len(df.columns)),
    "total_valores_nulos": int(df.isna().sum().sum()),
    "filas_con_al_menos_un_nulo": int(df.isna().any(axis=1).sum()),
    "filas_duplicadas_exactas": int(df.duplicated().sum()),
    "registros_con_problemas_temporales": int(
        len(problemas_temporales)
    ),
    "codigos_departamento_con_multiples_nombres": int(
        problemas_depto_codigo["Código Dane departamento"].nunique()
    ),
    "codigos_municipio_con_multiples_nombres": int(
        problemas_municipio_codigo["Código Dane municipio"].nunique()
    ),
    "combinaciones_repetidas_clave_1": int(
        resumen_clave_1["combinaciones_repetidas"]
    ),
    "combinaciones_repetidas_clave_2": int(
        resumen_clave_2["combinaciones_repetidas"]
    ),
    "registros_area_cosechada_mayor_sembrada": int(
        casos_area_cosechada_mayor.sum()
    ),
    "registros_sembrada_cero_cosechada_positiva": int(
        casos_sembrada_cero_cosechada_positiva.sum()
    ),
    "registros_con_metricas_negativas": int(
        casos_metricas_negativas.sum()
    ),
}

resumen_general_df = pd.DataFrame(
    list(resumen_general.items()),
    columns=["indicador", "valor"],
)

display(resumen_general_df)

with open(
    OUTPUT_DIR / "resumen_general.json",
    "w",
    encoding="utf-8",
) as archivo:
    json.dump(
        resumen_general,
        archivo,
        ensure_ascii=False,
        indent=4,
        default=str,
    )

resumen_general_df.to_csv(
    OUTPUT_DIR / "resumen_general.csv",
    index=False,
    encoding="utf-8-sig",
)

,indicador,valor
0,archivo_fuente,Evaluaciones_Agropecuarias_Municipales_–_EVA._...
1,total_registros,166732
2,total_columnas,18
3,total_valores_nulos,0
4,filas_con_al_menos_un_nulo,0
5,filas_duplicadas_exactas,0
6,registros_con_problemas_temporales,0
7,codigos_departamento_con_multiples_nombres,0
8,codigos_municipio_con_multiples_nombres,0
9,combinaciones_repetidas_clave_1,0


## 10. Conclusiones del perfilamiento

### 10.1 Estructura y completitud

El análisis estructural permitió establecer el volumen de registros y
columnas de la fuente, además de identificar la presencia de valores
nulos y duplicados exactos.

Los resultados cuantitativos se encuentran en los archivos generados
por el notebook. Cualquier diferencia frente a ejecuciones anteriores
deberá investigarse para determinar si corresponde a una actualización
legítima de la fuente o a una variación inesperada.

### 10.2 Métricas agrícolas

El análisis de las métricas permitió identificar sus rangos, dispersión,
valores extremos, ceros, negativos y posibles problemas de conversión.

Las reglas de calidad deberán considerar el significado de cada variable
y evitar la eliminación automática de valores únicamente por ser extremos.

### 10.3 Consistencia temporal

La distribución de Año y Periodo permite revisar los niveles de
agregación presentes en la fuente. La integración climática deberá
respetar las observaciones temporales originales y definir explícitamente
la agregación de las variables meteorológicas.

### 10.4 Consistencia geográfica

Los códigos DANE serán la referencia principal para relacionar los
registros agrícolas con el catálogo geográfico.

Las inconsistencias de códigos y nombres deberán resolverse o
documentarse antes de integrar las coordenadas utilizadas para consultar
la API climática.

### 10.5 Clasificación de cultivos

Las relaciones entre grupos, subgrupos, cultivos, desagregaciones y
códigos permitirán establecer qué atributos identifican cada entidad
y cuáles deben conservarse en la dimensión de cultivos.

### 10.6 Granularidad

La clave candidata deberá seleccionarse con base en los resultados
de unicidad, la presencia de componentes nulos y el significado
de cada registro. La ausencia de duplicados exactos no basta para
confirmar la granularidad.

### 10.7 Próximas etapas

Los resultados de este notebook servirán como insumo para:

1. Definir y justificar las reglas de calidad de la fuente EVA.
2. Preparar la extracción y el perfilamiento de Open-Meteo.
3. Validar el catálogo geográfico y las coordenadas municipales.
4. Definir el contrato de integración entre datos agrícolas y climáticos.
5. Refinar el modelo dimensional.
6. Implementar validaciones con Great Expectations.
7. Incorporar las verificaciones al pipeline automatizado de Airflow.

In [28]:
archivos_generados = sorted(
    OUTPUT_DIR.glob("*")
)

print(f"Directorio de resultados: {OUTPUT_DIR}")
print(f"Cantidad de archivos generados: {len(archivos_generados)}")

for archivo in archivos_generados:
    print(f"- {archivo.name}")

Directorio de resultados: c:\Users\Monica\Desktop\ETL\ODS2\outputs\profiling
Cantidad de archivos generados: 32
- consistencia_metricas.csv
- departamentos_codigo_multiples_nombres.csv
- departamentos_nombre_multiples_codigos.csv
- distribucion_ciclo_del_cultivo.csv
- distribucion_codigo_del_cultivo.csv
- distribucion_cultivo.csv
- distribucion_desagregacion_cultivo.csv
- distribucion_estado_físico_del_cultivo.csv
- distribucion_grupo_cultivo.csv
- distribucion_nombre_cientifico_del_cultivo.csv
- distribucion_subgrupo.csv
- duplicados_exactos.csv
- errores_conversion_numerica.csv
- evaluacion_granularidad.csv
- metricas_estadisticas.csv
- municipios_codigo_multiples_nombres.csv
- municipios_nombre_multiples_codigos.csv
- municipios_relaciones_inconsistentes.csv
- problemas_temporales.csv
- registros_para_investigar_metricas.csv
- relacion_cultivo__código_del_cultivo.csv
- relacion_cultivo__desagregación_cultivo.csv
- relacion_código_del_cultivo__cultivo.csv
- relacion_código_del_cultiv

In [29]:

# =========================================================
# EVA-DIVIPOLA MUNICIPALITY MATCH DIAGNOSTICS
# =========================================================

import pandas as pd

EVA_CODE = "Código Dane municipio"
EVA_DEPT_CODE = "Código Dane departamento"
EVA_DEPT = "Departamento"
EVA_MUNICIPALITY = "Municipio"

DIV_CODE = "Código Municipio"
DIV_DEPT_CODE = "Código Departamento"
DIV_DEPT = "Nombre Departamento"
DIV_MUNICIPALITY = "Nombre Municipio"
DIV_TYPE = "Tipo: Municipio / Isla / Área no municipalizada"

# ---------------------------------------------------------
# 1. Normalize codes consistently
# ---------------------------------------------------------

def normalize_code(series):
    codes = series.astype("string").str.strip()
    codes = codes.str.replace(r"\.0$", "", regex=True)
    return codes.str.zfill(5).mask(codes.eq(""))

eva_geo = eva[
    [EVA_DEPT_CODE, EVA_DEPT, EVA_CODE, EVA_MUNICIPALITY]
].copy()

div_geo = divipola_work[
    [
        DIV_DEPT_CODE,
        DIV_DEPT,
        DIV_CODE,
        DIV_MUNICIPALITY,
        DIV_TYPE,
        "Latitud",
        "longitud",
    ]
].copy()

eva_geo["municipality_code"] = normalize_code(eva_geo[EVA_CODE])
div_geo["municipality_code"] = normalize_code(div_geo[DIV_CODE])

# Reject malformed non-null codes rather than silently matching them.
eva_bad_codes = eva_geo[
    eva_geo["municipality_code"].notna()
    & ~eva_geo["municipality_code"].str.fullmatch(r"\d{5}", na=False)
]

div_bad_codes = div_geo[
    div_geo["municipality_code"].notna()
    & ~div_geo["municipality_code"].str.fullmatch(r"\d{5}", na=False)
]

print("Malformed non-null EVA codes:", len(eva_bad_codes))
print("Malformed non-null DIVIPOLA codes:", len(div_bad_codes))

# ---------------------------------------------------------
# 2. Check duplicate and conflicting municipality codes
# ---------------------------------------------------------

eva_unique = eva_geo.dropna(subset=["municipality_code"]).drop_duplicates()

eva_conflicts = (
    eva_unique.groupby("municipality_code")
    .agg(
        municipality_names=(EVA_MUNICIPALITY, "nunique"),
        department_codes=(EVA_DEPT_CODE, "nunique"),
        department_names=(EVA_DEPT, "nunique"),
    )
    .query(
        "municipality_names > 1 "
        "or department_codes > 1 "
        "or department_names > 1"
    )
)

div_valid = div_geo.dropna(subset=["municipality_code"]).copy()

div_duplicate_codes = div_valid[
    div_valid["municipality_code"].duplicated(keep=False)
].sort_values("municipality_code")

print("\nDistinct EVA municipality codes:",
      eva_unique["municipality_code"].nunique())
print("Distinct DIVIPOLA codes:",
      div_valid["municipality_code"].nunique())
print("DIVIPOLA codes with duplicate rows:",
      div_duplicate_codes["municipality_code"].nunique())
print("EVA codes with conflicting municipality attributes:",
      len(eva_conflicts))

display(eva_conflicts.head(20))
display(div_duplicate_codes.head(20))

# Stop before joining if the reference key is ambiguous.
if div_valid["municipality_code"].duplicated().any():
    raise ValueError(
        "DIVIPOLA contains duplicate municipality codes. "
        "Inspect the division types before building the crosswalk."
    )

if not eva_conflicts.empty:
    raise ValueError(
        "EVA has conflicting attributes for some municipality codes. "
        "Resolve these before building a one-row-per-municipality catalog."
    )

# ---------------------------------------------------------
# 3. Create one EVA row per municipality and join by code
# ---------------------------------------------------------

eva_unique = (
    eva_unique.sort_values("municipality_code")
    .drop_duplicates("municipality_code")
)

crosswalk = eva_unique.merge(
    div_valid,
    on="municipality_code",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("\nEVA-DIVIPOLA match status:")
display(
    crosswalk["_merge"]
    .value_counts()
    .rename_axis("match_status")
    .reset_index(name="municipalities")
)

# ---------------------------------------------------------
# 4. Validate coordinates and municipality classification
# ---------------------------------------------------------

lat = pd.to_numeric(crosswalk["Latitud"], errors="coerce")
lon = pd.to_numeric(crosswalk["longitud"], errors="coerce")

crosswalk["coordinates_valid"] = (
    lat.between(-5, 14)
    & lon.between(-82, -66)
)

crosswalk["is_official_municipality"] = (
    crosswalk[DIV_TYPE].eq("Municipio")
)

crosswalk["eligible_for_weather"] = (
    crosswalk["_merge"].eq("both")
    & crosswalk["coordinates_valid"]
    & crosswalk["is_official_municipality"]
)

print("\nGeographic coverage summary:")
print("EVA municipalities:", len(crosswalk))
print("Matched to DIVIPOLA:", crosswalk["_merge"].eq("both").sum())
print("Unmatched to DIVIPOLA:", crosswalk["_merge"].eq("left_only").sum())
print("Invalid or missing coordinates:",
      (~crosswalk["coordinates_valid"]).sum())
print("Not classified as Municipality:",
      (~crosswalk["is_official_municipality"]).sum())
print("Eligible for weather extraction:",
      crosswalk["eligible_for_weather"].sum())

print("\nEVA municipalities requiring review:")
display(
    crosswalk.loc[
        ~crosswalk["eligible_for_weather"],
        [
            "municipality_code",
            EVA_MUNICIPALITY,
            EVA_DEPT,
            DIV_MUNICIPALITY,
            DIV_TYPE,
            "Latitud",
            "longitud",
            "_merge",
            "coordinates_valid",
        ],
    ]
)

# Keep the complete crosswalk for auditing.
# Do not replace municipality_locations until discrepancies are reviewed.


NameError: name 'eva' is not defined

In [30]:

from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# 1. Paths and validation parameters
# ---------------------------------------------------------

BATCH_DIR = PROJECT_ROOT / "data" / "raw" / "open_meteo_daily_batches"

START_DATE = pd.Timestamp("2019-01-01")
END_DATE = pd.Timestamp("2025-12-31")
EXPECTED_DAYS = len(pd.date_range(START_DATE, END_DATE, freq="D"))

print(f"Batch directory: {BATCH_DIR}")
print(f"Directory exists: {BATCH_DIR.is_dir()}")

if not BATCH_DIR.is_dir():
    raise FileNotFoundError(f"Batch directory not found: {BATCH_DIR}")

# Use the existing catalog and its established batch assignment.
catalog = municipality_locations.copy()

if "batch_id" not in catalog.columns:
    catalog = catalog.sort_values("municipality_code").reset_index(drop=True)
    catalog["batch_id"] = catalog.index // 25 + 1

catalog["municipality_code"] = (
    catalog["municipality_code"].astype("string").str.strip().str.zfill(5)
)

expected_dates = pd.date_range(START_DATE, END_DATE, freq="D")
results = []

# ---------------------------------------------------------
# 2. Validate each existing batch
# ---------------------------------------------------------

for batch_id in range(1, 4):
    path = BATCH_DIR / f"weather_batch_{batch_id:03d}.csv"

    if not path.is_file():
        print(f"\nMISSING: {path.name}")
        continue

    df = pd.read_csv(path, dtype={"municipality_code": "string"})

    # Normalize identifiers and dates.
    df["municipality_code"] = (
        df["municipality_code"].astype("string").str.strip().str.zfill(5)
    )
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    expected_codes = set(
        catalog.loc[catalog["batch_id"].eq(batch_id), "municipality_code"]
    )
    actual_codes = set(df["municipality_code"].dropna().unique())

    # Verify the actual municipality/date combinations.
    duplicate_pairs = int(
        df.duplicated(["municipality_code", "date"]).sum()
    )

    per_municipality = df.groupby("municipality_code")["date"].agg(
        records="size",
        distinct_dates="nunique",
        first_date="min",
        last_date="max",
    )

    complete_series = (
        len(actual_codes) == len(expected_codes)
        and actual_codes == expected_codes
        and len(per_municipality) == len(expected_codes)
        and per_municipality["records"].eq(EXPECTED_DAYS).all()
        and per_municipality["distinct_dates"].eq(EXPECTED_DAYS).all()
        and per_municipality["first_date"].eq(START_DATE).all()
        and per_municipality["last_date"].eq(END_DATE).all()
        and duplicate_pairs == 0
    )

    missing_codes = sorted(expected_codes - actual_codes)
    unexpected_codes = sorted(actual_codes - expected_codes)

    print(f"\n--- Batch {batch_id:03d} ---")
    print(f"File: {path.name}")
    print(f"Rows: {len(df):,}")
    print(f"Expected municipalities: {len(expected_codes)}")
    print(f"Actual municipalities: {len(actual_codes)}")
    print(f"Expected rows: {len(expected_codes) * EXPECTED_DAYS:,}")
    print(f"Duplicate municipality-date pairs: {duplicate_pairs}")
    print(f"Missing municipalities: {missing_codes}")
    print(f"Unexpected municipalities: {unexpected_codes}")
    print(f"Complete date series: {complete_series}")

    results.append({
        "batch_id": batch_id,
        "file": path.name,
        "rows": len(df),
        "expected_rows": len(expected_codes) * EXPECTED_DAYS,
        "municipalities": len(actual_codes),
        "expected_municipalities": len(expected_codes),
        "duplicate_pairs": duplicate_pairs,
        "missing_codes": len(missing_codes),
        "unexpected_codes": len(unexpected_codes),
        "complete": complete_series,
    })

validation_results = pd.DataFrame(results)

print("\n--- Validation summary ---")
display(validation_results)


Batch directory: c:\Users\Monica\Desktop\ETL\ODS2\data\raw\open_meteo_daily_batches
Directory exists: True


NameError: name 'municipality_locations' is not defined

In [31]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "data" / "source").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("No se encontró la raíz del proyecto.")
    PROJECT_ROOT = PROJECT_ROOT.parent

source_dir = PROJECT_ROOT / "data" / "source"

eva = pd.read_csv(
    source_dir / "Evaluaciones_Agropecuarias_Municipales_–_EVA._2019_-_2025._Base_Agrícola_20260905.csv",
    nrows=0,
)

divipola = pd.read_csv(
    source_dir / "divipola.csv",
    nrows=0,
)

print("EVA:")
print(eva.columns.tolist())

print("\nDIVIPOLA:")
print(divipola.columns.tolist())

EVA:
['Código Dane departamento', 'Departamento', 'Código Dane municipio', 'Municipio', 'Grupo cultivo', 'Subgrupo', 'Cultivo', 'Desagregación cultivo', 'Año', 'Periodo', 'Área sembrada', 'Área cosechada', 'Producción', 'Rendimiento', 'Ciclo del cultivo', 'Estado físico del cultivo', 'Código del cultivo', 'Nombre científico del cultivo']

DIVIPOLA:
['Código Departamento', 'Nombre Departamento', 'Código Municipio', 'Nombre Municipio', 'Tipo: Municipio / Isla / Área no municipalizada', 'longitud', 'Latitud']


In [33]:
import sys
from pathlib import Path

# Locate the project root
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src" / "extraction").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError(
            "Could not locate src/extraction/."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

# Make the extraction module importable
sys.path.insert(0, str(PROJECT_ROOT / "src" / "extraction"))

from open_meteo import (
    load_municipality_catalog,
    normalize_code,
)

print("Extraction functions imported successfully.")

ImportError: cannot import name 'load_municipality_catalog' from 'open_meteo' (C:\Users\Monica\Desktop\ETL\ODS2\src\extraction\open_meteo.py)